<a href="https://colab.research.google.com/github/tarasovEgor/DeepLearningKurs/blob/main/src/lab_1/cnn_clf_lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 1.**

---

## Классификация изображений с использованием CNN, предобработка, аугментация и анализ метрик.

---

## **Цель работы:** изучить основы сверточных нейронных сетей (CNN) в PyTorch, исследовать влияние предобработки, аугментации, архитектуры сети и гиперпараметров на качество классификации изображений.

### 0. Init steps

---

#### Импорты библиотек:

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CyclicLR

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

Устройство: cuda


#### Вспомогательные функции:

---

##### 0.1 Загрузка данных:

In [4]:
def get_transforms(dataset_name, augment=False):
    if dataset_name == "CIFAR10":
        if augment:
            train_transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, padding=4),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
        else:
            train_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    elif dataset_name == "FashionMNIST":
        if augment:
            train_transform = transforms.Compose([
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])
        else:
            train_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    elif dataset_name == "SVHN":
        if augment:
            train_transform = transforms.Compose([
                transforms.ColorJitter(brightness=0.2),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
        else:
            train_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    return train_transform, test_transform


def get_dataloaders(dataset_name, batch_size=64, augment=False):
    train_transform, test_transform = get_transforms(dataset_name, augment)

    if dataset_name == "CIFAR10":
        train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
        test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
        in_channels = 3
        img_size = 32
        num_classes = 10

    elif dataset_name == "FashionMNIST":
        train_data = datasets.FashionMNIST(root="./data", train=True, download=True, transform=train_transform)
        test_data = datasets.FashionMNIST(root="./data", train=False, download=True, transform=test_transform)
        in_channels = 1
        img_size = 28
        num_classes = 10

    elif dataset_name == "SVHN":
        train_data = datasets.SVHN(root="./data", split="train", download=True, transform=train_transform)
        test_data = datasets.SVHN(root="./data", split="test", download=True, transform=test_transform)
        in_channels = 3
        img_size = 32
        num_classes = 10

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, in_channels, img_size, num_classes

##### 0.2 Базовая CNN:

In [5]:
class BasicCNN(nn.Module):
    def __init__(
        self,
        in_channels=3,
        num_classes=10,
        img_size=32,
        filters=(32, 64),
        kernel_sizes=(3, 3),
        use_bn=False,
        dropout=0.0,
        pool_type="max"
    ):
        super().__init__()

        k1, k2 = kernel_sizes
        p1 = k1 // 2
        p2 = k2 // 2

        self.conv1 = nn.Conv2d(in_channels, filters[0], kernel_size=k1, padding=p1)
        self.conv2 = nn.Conv2d(filters[0], filters[1], kernel_size=k2, padding=p2)

        self.use_bn = use_bn
        self.bn1 = nn.BatchNorm2d(filters[0])
        self.bn2 = nn.BatchNorm2d(filters[1])

        if pool_type == "max":
            self.pool = nn.MaxPool2d(2, 2)
        else:
            self.pool = nn.AvgPool2d(2, 2)

        self.dropout = nn.Dropout(dropout)

        # После двух pooling размер уменьшается в 4 раза
        feature_size = img_size // 4
        self.fc1 = nn.Linear(filters[1] * feature_size * feature_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        if self.use_bn:
            x = self.bn1(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        if self.use_bn:
            x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

##### 0.3 Более глубокая CNN для 4-й части:

In [6]:
class DeeperCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, img_size=32, dropout=0.3):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout)

        feature_size = img_size // 8
        self.fc1 = nn.Linear(128 * feature_size * feature_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

##### 0.4 Оценка модели:

In [7]:
def evaluate_model(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = {
        "loss": total_loss / len(loader),
        "acc": correct / total,
        "precision": precision_score(all_labels, all_preds, average="weighted", zero_division=0),
        "recall": recall_score(all_labels, all_preds, average="weighted", zero_division=0),
        "f1": f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    }

    try:
        metrics["roc_auc"] = roc_auc_score(all_labels, np.array(all_probs), multi_class="ovr")
    except:
        metrics["roc_auc"] = None

    return metrics

##### 0.5 Обучение модели:

In [8]:
def train_model(
    model,
    train_loader,
    test_loader,
    epochs=5,
    lr=0.001,
    optimizer_name="adam",
    scheduler_type=None,
    accumulation_steps=1
):
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)

    scheduler = None
    if scheduler_type == "cyclic":
        scheduler = CyclicLR(
            optimizer,
            base_lr=1e-4,
            max_lr=1e-2,
            step_size_up=max(1, len(train_loader) // 2),
            cycle_momentum=(optimizer_name == "sgd")
        )

    history = {
        "train_loss": [],
        "test_loss": [],
        "train_acc": [],
        "test_acc": []
    }

    for epoch in range(epochs):
        model.train()

        running_loss = 0
        correct = 0
        total = 0

        optimizer.zero_grad()

        for step, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels) / accumulation_steps
            loss.backward()

            if (step + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

                if scheduler is not None:
                    scheduler.step()

            running_loss += loss.item() * accumulation_steps
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        if len(train_loader) % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        test_metrics = evaluate_model(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_metrics["loss"])
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_metrics["acc"])

        print(
            f"Эпоха {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Test Acc: {test_metrics['acc']:.4f}"
        )

    final_test_metrics = evaluate_model(model, test_loader, criterion)
    final_train_metrics = evaluate_model(model, train_loader, criterion)

    return history, final_train_metrics, final_test_metrics

##### 0.6 Визуализация:

In [10]:
def plot_history(history, title="Графики обучения"):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["test_loss"], label="Test Loss")
    plt.xlabel("Эпоха")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["test_acc"], label="Test Accuracy")
    plt.xlabel("Эпоха")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()

    plt.show()


def print_metrics(name, metrics):
    print(f"\n{name}")
    print(f"Loss      : {metrics['loss']:.4f}")
    print(f"Accuracy  : {metrics['acc']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"F1-score  : {metrics['f1']:.4f}")
    if metrics["roc_auc"] is not None:
        print(f"ROC-AUC   : {metrics['roc_auc']:.4f}")
    else:
        print("ROC-AUC   : не удалось вычислить")

### Часть 1. Базовая CNN для CIFAR-10

#### 1.1 Подготовка данных:

In [11]:
train_loader, test_loader, in_channels, img_size, num_classes = get_dataloaders(
    dataset_name="CIFAR10",
    batch_size=64,
    augment=False
)

100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


#### 1.2 Определение архитектуры:

In [12]:
model = BasicCNN(
    in_channels=in_channels,
    num_classes=num_classes,
    img_size=img_size,
    filters=(32, 64),
    kernel_sizes=(3, 3),
    use_bn=False,
    dropout=0.0,
    pool_type="max"
).to(device)

#### 1.3 Обучение:

In [13]:
history_part1, train_metrics_part1, test_metrics_part1 = train_model(
    model,
    train_loader,
    test_loader,
    epochs=10,
    lr=0.001,
    optimizer_name="adam"
)

Эпоха 1/10 | Train Loss: 1.3238 | Train Acc: 0.5272 | Test Acc: 0.6324
Эпоха 2/10 | Train Loss: 0.9415 | Train Acc: 0.6693 | Test Acc: 0.6776
Эпоха 3/10 | Train Loss: 0.7956 | Train Acc: 0.7199 | Test Acc: 0.7027
Эпоха 4/10 | Train Loss: 0.6847 | Train Acc: 0.7611 | Test Acc: 0.7246
Эпоха 5/10 | Train Loss: 0.5887 | Train Acc: 0.7938 | Test Acc: 0.7302



KeyboardInterrupt



### 1.4 Визуализация и метрики: